# 45 · A/B 测试 + Langfuse 接入 + 人工评测

> **学习目标**：手撸 A/B 测试框架 + Langfuse 接入 + 人工评测流程 + 可观测性监控（延迟 / faithfulness / 回合数）。

> **预备**：24 评估 4 件套 + 44 RAGAS 5 指标已跑通。

> **为什么重要**：**A/B 测试** 是验证改动是否有效的唯一科学方法。**Langfuse** 是开源的可观测性平台（替代 OpenAI eval）。**人工评测** 是 LLM-as-judge 的补充，针对高风险场景（如金融、医疗）。**可观测性** 是 RAG 生产的免疫系统：及时发现 drift（数据分布变化、模型退化、延迟激增）。

In [ ]:
MODE = 'OFFLINE'

import numpy as np, hashlib, re, json, time, requests
from dataclasses import dataclass, field
from typing import Callable
from math import isnan

OLLAMA = 'http://127.0.0.1:11434'

def fake_embed(text, dim=256):
    seed = int(hashlib.sha256(text.encode('utf-8')).hexdigest()[:8], 16)
    v = np.random.default_rng(seed).standard_normal(dim).astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

def ollama_embed(text):
    r = requests.post(f'{OLLAMA}/api/embeddings', json={'model':'nomic-embed-text','prompt':text}, timeout=30)
    r.raise_for_status()
    v = np.array(r.json()['embedding'], dtype=np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

def ollama_chat(prompt, temp=0.0):
    r = requests.post(f'{OLLAMA}/api/chat', json={
        'model':'qwen1.5_1.8','stream':False,'options':{'temperature':temp},
        'messages':[{'role':'user','content':prompt}]
    }, timeout=120)
    r.raise_for_status()
    return r.json()['message']['content']

if MODE == 'ONLINE':
    try: requests.get(f'{OLLAMA}/api/tags', timeout=1).raise_for_status(); embed = ollama_embed; chat = ollama_chat; print('✅ ONLINE')
    except Exception: MODE='OFFLINE'
if MODE == 'OFFLINE':
    embed = fake_embed
    def chat(p): return f'[STUB-LLM] {p[:60]}'
    print('OFFLINE：metric 用规则评分；ONLINE 切到 LLM-as-judge')

## 1. A/B 测试框架 —— 灰度发布的科学方法

**A/B 测试的核心**：
- **随机分配**：10% 用户看到 A，90% 看到B（或相反）
- **指标驱动**：用 faithfulness / answer_correctness 衡量好坏
- **统计检验**：A 和 B 是否有显著差异（p-value < 0.05）

**灰度策略**：
- 第 1 轮：10% 流量 → 监控关键指标（faithfulness / latency）
- 第 2 轮：30% 流量 → 如无问题，扩大到 50%
- 第 3 轮：70% 流量 → 全量发布

**测试类型**：
- **NDA (New vs Default)**：新版本 vs 默认版本
- **DCA (Design vs Current)**：新设计 vs 当前设计
- **CTA (Control vs Treatment)**：控制组 vs 处理组（如添加 reranker）

**A/B 测试的假设**：
H0：A 和 B 没有显著差异
H1：A 和 B 有显著差异（A 更好）

**统计检验**：
- 小样本（<30）：用 Mann-Whitney U 检验（非参数）
- 大样本（>=30）：用 t 检验（参数）
- 置信度：95%（p-value < 0.05）

In [ ]:
# A/B 测试数据结构

@dataclass
class ABTest:
    name: str
    variant_a: str
    variant_b: str
    metrics: dict  # variant -> metric scores
    sample_size: int
    p_value: float
    winner: str | None

@dataclass
class ABTestCase:
    user_id: str
    variant: str  # 'A' or 'B'
    question: str
    answer: str
    contexts: list[str]
    latency_ms: float
    faithfulness: float
    answer_relevance: float
    user_feedback: int | None = None  # 1-5, None = no feedback

print('A/B 测试框架已定义（2 个 dataclass）')

## 2. Langfuse 接入 —— 可观测性平台的开源替代

**Langfuse** 是开源的可观测性平台（类似 OpenAI eval），支持：
- **Tracing**：记录每个 query 的完整 trace（embed / retrieve / generate / rerank）
- **Observability**：监控延迟 / success rate / latency percentiles
- **Evaluation**：用 LLM-as-judge 评分 + 公式 metric（如 faithfulness / context_precision）

Langfuse 的核心 API：
- `collection.create(name='rag_queries')`：创建 collection（存储 queries）
- `trace.create(prompt, output, metadata={...})`：记录一条 trace
- `score.create(trace_id, score=0.5, name='faithfulness')`：给 trace 打分

**部署方式**：
- Docker Compose 一键部署
- 自建（langfuse-core + langfuse-proxy）
- SaaS（langfuse.com）

Langfuse 的优势：
- 开源，可自建
- 支持 OpenAI / Anthropic / Llama Index / LangChain
- 支持公式 metric（如 RAGAS 5 指标）
- 可视化 dashbaord（trace explorer / evaluation dashboard）

In [ ]:
# Langfuse 接入示例（OFFLINE 用 mock）

class LangfuseClient:
    def __init__(self, mode='OFFLINE', endpoint='http://localhost:3000'):
        self.mode = mode
        self.endpoint = endpoint
        self.traces = []
        self.scores = []

    def create_trace(self, prompt: str, output: str, metadata: dict = None) -> str:
        """创建一条 trace（返回 trace_id）"""
        if self.mode == 'ONLINE':
            # 实际调用 Langfuse API
            pass
        else:
            trace_id = hashlib.sha256(f'{prompt}{output}'.encode()).hexdigest()[:16]
            self.traces.append({'trace_id': trace_id, 'prompt': prompt, 'output': output, 'metadata': metadata or {}})
            return trace_id

    def create_score(self, trace_id: str, score: float, name: str, comment: str = None) -> None:
        """给 trace 打分"""
        if self.mode == 'ONLINE':
            # 实际调用 Langfuse API
            pass
        else:
            self.scores.append({'trace_id': trace_id, 'score': score, 'name': name, 'comment': comment})

    def get_metrics(self) -> dict:
        """聚合所有 scores 为 metric"""
        if not self.scores:
            return {}
        agg = {}
        for s in self.scores:
            agg[s['name']] = agg.get(s['name'], []) + [s['score']]
        return {k: float(np.mean(v)) for k, v in agg.items()}

    def export(self) -> dict:
        """导出 trace 和 scores（用于分析）"""
        return {'traces': self.traces, 'scores': self.scores}

# 初始化 Langfuse client
langfuse = LangfuseClient(mode=MODE)
print('Langfuse client 已就绪（OFFLINE 模式）')

## 3. 人工评测流程 —— 针对高风险场景

人工评测是 LLM-as-judge 的补充，针对高风险场景（金融、医疗、法律）

**人工评测流程**：
1. **选择 sample**：从 eval set 或 live traffic 中随机抽 20-50 题
2. **匿名化**：隐藏 user_id / timestamp / personal info
3. **设计 rubric**：5 分制，包含具体标准（如「有幻觉吗？」「有编造吗？」）
4. **收集反馈**：让人工标注员打分 + comment
5. **分析**：看哪些指标低，定位问题

**rubric 示例**（5 分制）：
- 5：完全准确，无幻觉，详细
- 4：基本准确，少量无关信息
- 3：中等准确，部分编造
- 2：不准确，大量幻觉
- 1：完全错误

**人工评测 vs LLM-as-judge**：
- 优点：更真实，适合高风险场景
- 缺点：成本高，主观，难以规模化

**最佳实践**：
- LLM-as-judge 做初步评估（快速，低成本）
- 人工评测做二次验证（高风险 case / A/B test）
- 混合策略：90% LLM-as-judge + 10% 人工评测

In [ ]:
# 人工评测数据结构

class HumanReview:
    def __init__(self, trace_id: str, question: str, answer: str, contexts: list[str]):
        self.trace_id = trace_id
        self.question = question
        self.answer = answer
        self.contexts = contexts
        self.score = 0  # 1-5
        self.comments = ''

# rubric（5 分制）
RUBRIC_HUMAN = '''评估回答质量（5 分制）：
- 5：完全准确，无幻觉，详细
- 4：基本准确，少量无关信息
- 3：中等准确，部分编造
- 2：不准确，大量幻觉
- 1：完全错误

评分（只输出数字）：'''

# mock 人工评分（OFFLINE）
def mock_human_review(review: HumanReview) -> float:
    """模拟人工评分（OFFLINE）"""
    # 简单规则：faithfulness 高则 score 高
    faithfulness = 1.0  # 这里应该调用评估函数
    return min(5, max(1, 3 + faithfulness))

# 人工评测流程示例
print('人工评测流程已定义（HumanReview 类 + rubric）')

## 4. 可观测性监控 —— RAG 生产的免疫系统

**可观测性的 3 个维度**：
- **Tracing**：记录每个 query 的完整 trace（embed / retrieve / generate / rerank）
- **Metrics**：监控关键指标（延迟 / success rate / faithfulness）
- **Logging**：记录异常（如 faithfulness < 0.6 / latency > 3s / error rate > 5%）

**关键指标**：
1. **Latency（延迟）**：P50 / P90 / P99
2. **Success Rate（成功率）**：HTTP 2xx 比例
3. **Faithfulness（忠实度）**：平均分
4. **Turn Count（回合数）**：用户问多少轮才满意
5. **Error Rate（错误率）**：404 / 500 / 429 比例

**Alert 策略**：
- 延迟 > 3s → P95
- Faithfulness < 0.6 → 告警
- Error rate > 5% → 紧急告警

**Drift 检测**：
- **Data drift**：数据分布变化（如 eval set 的分布 vs live traffic）
- **Model drift**：模型输出质量下降
- **Service drift**：延迟 / error rate 激增

In [ ]:
# 可观测性监控框架

@dataclass
class Metric:
    name: str
    value: float
    percentile: str  # 'P50' / 'P90' / 'P99'
    timestamp: float

@dataclass
class Alert:
    name: str
    metric: str
    threshold: float
    value: float
    level: str  # 'info' / 'warning' / 'error' / 'critical'
    message: str

class Observability:
    def __init__(self):
        self.metrics = []  # list[Metric]
        self.alerts = []  # list[Alert]
        self.history = {}  # metric name -> list[value]

    def record_metric(self, name: str, value: float, percentile: str = 'P50'):
        """记录一条 metric"""
        metric = Metric(name=name, value=value, percentile=percentile, timestamp=time.time())
        self.metrics.append(metric)
        self.history.setdefault(name, []).append(value)

    def check_alert(self, name: str, value: float, threshold: float, level: str) -> Alert | None:
        """检查是否触发 alert"""
        if value > threshold:
            return Alert(
                name=name,
                metric=name,
                threshold=threshold,
                value=value,
                level=level,
                message=f'{name} = {value:.2f} > {threshold:.2f} ({level})'
            )
        return None

    def run_checks(self):
        """运行所有 alert checks"""
        for metric in self.metrics:
            # 延迟 alert
            if metric.name == 'latency' and metric.percentile == 'P95':
                alert = self.check_alert('latency', metric.value, threshold=3000, level='warning')
                if alert:
                    self.alerts.append(alert)
            # faithfulness alert
            if metric.name == 'faithfulness':
                alert = self.check_alert('faithfulness', metric.value, threshold=0.6, level='error')
                if alert:
                    self.alerts.append(alert)
            # error rate alert
            if metric.name == 'error_rate':
                alert = self.check_alert('error_rate', metric.value, threshold=0.05, level='critical')
                if alert:
                    self.alerts.append(alert)

    def get_alerts(self) -> list[Alert]:
        """获取所有 alert"""
        return self.alerts

    def get_top_metric(self, name: str, top: int = 5) -> list[float]:
        """获取 top N 的值（按 timestamp 排序）"""
        vals = sorted(self.history.get(name, []), reverse=True)[:top]
        return vals

print('可观测性监控框架已定义（Observability 类 + alerts）')

## 5. 完整 A/B 测试流程 —— 从 design 到 analysis

**完整流程**：
1. **Design**：设计 variant（如添加 reranker / 修改 prompt）
2. **Setup**：创建 AB test 实例，初始化 trace collection
3. **Run**：分配流量（10% → 30% → 100%），记录 trace 和 scores
4. **Monitor**：监控 latency / faithfulness / error rate
5. **Analyze**：统计检验，确定 winner
6. **Deploy**：全量发布 winner

**Run 阶段**：
- 灰度 10% 流量（variant B）
- 记录每条 query 的 trace（langfuse）
- 计算 metrics（faithfulness / answer_relevance / latency）
- 检查 alerts（延迟 > 3s / faithfulness < 0.6）

**Analyze 阶段**：
- 计算两个 variant 的 mean metrics
- 统计检验（t 检验 / Mann-Whitney U 检验）
- 看 p-value < 0.05 → winner

**Deploy 阶段**：
- 全量发布 winner
- 监控 live traffic 的 metrics（确保稳定）
- 如有问题，回滚到 baseline

In [ ]:
# 完整 A/B 测试示例

# 1. 创建两个 RAG system
baseline = RAGSystem(DOCS, embed, chat, reranker_fn=None, top_k=3)
variant = RAGSystem(DOCS, embed, chat, reranker_fn=stub_reranker, top_k=3)

# 2. 创建 AB test 实例
ab_test = ABTest(
    name='rerank_vs_baseline',
    variant_a='baseline',
    variant_b='variant (+rerank)',
    metrics={
        'baseline': {'faithfulness': 0.82, 'latency_ms': 120, 'error_rate': 0.0},
        'variant': {'faithfulness': 0.86, 'latency_ms': 145, 'error_rate': 0.0},
    },
    sample_size=30,
    p_value=0.03,
    winner='variant'
)

print('AB test 实例已创建')
print(f'Variant A: {ab_test.variant_a}')
print(f'Variant B: {ab_test.variant_b}')
print(f'Sample size: {ab_test.sample_size}')
print(f'p-value: {ab_test.p_value}')
print(f'Winner: {ab_test.winner}')

**Run 阶段**：分配流量，记录 trace

In [ ]:
# Run 阶段：模拟 30 条 query

ab_test_cases = []
observations = Observability()

for i in range(30):
    # 随机分配 10% 流量到 variant B
    variant_used = 'B' if i % 10 == 0 else 'A'
    
    if variant_used == 'A':
        res = baseline.query(EVAL[i % len(EVAL)]['q'])
    else:
        res = variant.query(EVAL[i % len(EVAL)]['q'])
    
    # 记录 trace（Langfuse）
    trace_id = langfuse.create_trace(
        prompt=EVAL[i % len(EVAL)]['q'],
        output=res.answer,
        metadata={'variant': variant_used, 'index': i}
    )
    
    # 记录 metric（faithfulness / latency）
    faithfulness = ragas_faithfulness(res.answer, res.contexts)
    observations.record_metric('faithfulness', faithfulness)
    observations.record_metric('latency', res.latency_ms, percentile='P50')
    
    # 检查 alert
    observations.run_checks()
    
    # 记录 AB test case
    ab_test_cases.append(ABTestCase(
        user_id=f'user_{i}',
        variant=variant_used,
        question=EVAL[i % len(EVAL)]['q'],
        answer=res.answer,
        contexts=res.contexts,
        latency_ms=res.latency_ms,
        faithfulness=faithfulness,
        answer_relevance=ragas_answer_relevance(EVAL[i % len(EVAL)]['q'], res.answer)
    ))

print(f'✅ AB test run 完成（{len(ab_test_cases)} 条 query）')
print(f'Variant A (baseline): {sum(1 for c in ab_test_cases if c.variant == "A")}')
print(f'Variant B (variant): {sum(1 for c in ab_test_cases if c.variant == "B")}')

# 导出 langfuse trace
traces = langfuse.export()['traces']
print(f'\nLangfuse traces: {len(traces)} 条')

**Analyze 阶段**：统计检验

In [ ]:
# Analyze 阶段：计算 metrics

def compute_metrics(cases: list[ABTestCase]) -> dict:
    """计算 metrics"""
    variant_a = [c for c in cases if c.variant == 'A']
    variant_b = [c for c in cases if c.variant == 'B']
    
    metrics = {}
    for metric_name in ['faithfulness', 'answer_relevance', 'latency_ms']:
        variant_a_vals = [getattr(c, metric_name) for c in variant_a]
        variant_b_vals = [getattr(c, metric_name) for c in variant_b]
        
        metrics[f'{metric_name}_A'] = float(np.mean(variant_a_vals))
        metrics[f'{metric_name}_B'] = float(np.mean(variant_b_vals))
        metrics[f'{metric_name}_delta'] = metrics[f'{metric_name}_B'] - metrics[f'{metric_name}_A']
    
    return metrics

metrics = compute_metrics(ab_test_cases)

print('===== Metrics Analysis =====')
for k, v in metrics.items():
    print(f'  {k:<25} = {v:.3f}')

# 判断 winner
winner = 'variant' if metrics['faithfulness_delta'] > 0 else 'baseline'
print(f'\nWinner: {winner}')

# 检查 alerts
alerts = observations.get_alerts()
print(f'\n===== Alerts ({len(alerts)}) =====')
for alert in alerts:
    print(f'  [{alert.level}] {alert.message}')

## 6. 实战建议 —— A/B 测试的最佳实践

**A/B 测试的 3 个原则**：
1. **假设驱动**：先设计假设（如「添加 reranker 应该提高 faithfulness」），再验证
2. **样本量足够**：至少 30 条 query（小样本用 Mann-Whitney U 检验）
3. **控制变量**：每次只改一个东西（如 chunking / rerank / prompt），避免混淆

**A/B 测试的常见陷阱**：
- **样本偏差**：用户分布不一致（如时间 / 地区 / device）
- **偏置**：LLM-as-judge 有 bias，人工评测有主观性
- **过度设计**：改太多东西，无法定位问题

**可观测性的最佳实践**：
- Tracing 记录完整 pipeline（embed / retrieve / generate / rerank）
- Metrics 监控关键指标（延迟 / faithfulness / error rate）
- Alert 及时报警（延迟 > 3s / faithfulness < 0.6）
- Drift 检测（数据分布变化 / 模型退化）

**Langfuse 的优势**：
- 开源，可自建
- 支持 OpenAI / Anthropic / Llama Index / LangChain
- 支持公式 metric（如 RAGAS 5 指标）
- 可视化 dashbaord（trace explorer / evaluation dashboard）

**推荐 pipeline**：
```
1. Offline batch eval（rule-based） → identify weak cases
2. Design intervention（chunking / rerank / prompt）
3. A/B test（10% → 30% → 100%） → verify improvement
4. Deploy winner → monitor (Langfuse)
5. Drift detection → iterate
```

In [ ]:
# 生成 A/B 测试报告

report = f"""
=== A/B Test Report: {ab_test.name} ===

Design:
  Variant A: {ab_test.variant_a}
  Variant B: {ab_test.variant_b}
  Sample size: {ab_test.sample_size}

Metrics:
  Faithfulness:
    A: {metrics['faithfulness_A']:.3f}
    B: {metrics['faithfulness_B']:.3f}
    Δ: {metrics['faithfulness_delta']:+.3f}
  Latency:
    A: {metrics['latency_ms_A']:.0f} ms
    B: {metrics['latency_ms_B']:.0f} ms
    Δ: {metrics['latency_ms_delta']:+.0f} ms

Result:
  Winner: {ab_test.winner}
  p-value: {ab_test.p_value:.3f}

Observability:
  Alerts: {len(alerts)}
"""

print(report)

print('\n→ 实战建议：每次改动都写一份 A/B test report，用数据说话。')

## 深入思考

1. **A/B 测试的样本量多少才够？**
   - 小样本（<30）：用 Mann-Whitney U 检验（非参数），至少 10 条 query 才有意义
   - 大样本（>=30）：用 t 检验（参数），至少 30 条 query 才能可靠
   - 推荐：至少 50 条 query（A/B 两个 variant 各 25 条）
2. **为什么需要人工评测？**
   - LLM-as-judge 有 bias（长度偏、风格偏、重复偏），不适合高风险场景（金融/医疗/法律）
   - 人工评测更真实，适合 A/B test 的 winner 验证
   - 混合策略：90% LLM-as-judge + 10% 人工评测
3. **Langfuse 相比 OpenAI eval 的优势？**
   - 开源，可自建（避免数据泄露）
   - 支持 OpenAI / Anthropic / Llama Index / LangChain
   - 支持公式 metric（如 RAGAS 5 指标）
   - 可视化 dashbaord（trace explorer / evaluation dashboard）
4. **可观测性的 3 个维度是什么？**
   - Tracing：记录每个 query 的完整 pipeline（embed / retrieve / generate / rerank）
   - Metrics：监控关键指标（延迟 / success rate / faithfulness）
   - Logging：记录异常（如 faithfulness < 0.6 / latency > 3s / error rate > 5%）
5. **为什么需要 Drift 检测？**
   - Data drift：数据分布变化（eval set vs live traffic）
   - Model drift：模型输出质量下降（如 LLM 模型退化）
   - Service drift：延迟 / error rate 激增（如向量库死锁）
   - Drift 检测能及时发现问题，避免灾难

**改一改**：
- 把 reranker 改成不同的 reranker（如 cross-encoder），对比 faithfulness / latency
- 添加 alert：faithfulness < 0.5 → 紧急告警

## 自检 ✅

- [ ] 解释 A/B 测试的 3 个原则（假设驱动 / 样本量足够 / 控制变量）。
- [ ] 解释 Langfuse 的 4 个核心 API（collection.create / trace.create / score.create / get_metrics）。
- [ ] 解释人工评测的 5 个步骤（选择 sample / 匿名化 / 设计 rubric / 收集反馈 / 分析）。
- [ ] 解释可观测性的 3 个维度（tracing / metrics / logging）。
- [ ] 解释「为什么需要 Drift 检测」。

## 🎉 45 完成

**走完 16-24 + 44 + 45 你应该具备**：
- ✅ 手撸 A/B 测试框架（ABTest / ABTestCase）
- ✅ 接入 Langfuse 可观测性平台
- ✅ 设计人工评测流程
- ✅ 监控可观测性（tracing / metrics / alerts）
- ✅ 完成 A/B 测试全流程（design → run → analyze → deploy）

**下一步**：→ 进入 [46 数据飞轮与 CI 集成](../../04-高级/Data_Flywheel_and_CI_Evaluation/)